©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、PyTorchを用いた転移学習・ファインチューニングを実践します。ImageNetで事前学習済みのモデルをベースに、CIFAR-10データセットへの適用を行い、学習済み重みを活用することで少ないデータ・短い学習時間でも高精度を達成できることを確認します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
%%capture
!pip uninstall matplotlib -y
!pip install matplotlib==3.8.2

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.8.1.78

# !pip uninstall tensorflow -y
# !pip install tensorflow==2.15.0

!pip uninstall torch -y
!pip install torch==2.1.1

!pip uninstall torchvision -y
!pip install torchvision==0.16.1

# Cifar10のデータを用いたファインチューニング

In [ ]:
# ライブラリのインポート
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
# 画像の前処理を定義:
# 1. 画像をPyTorchのテンソルに変換
# 2. 画像を正規化（平均: 0.5, 標準偏差: 0.5で各チャンネルを正規化）
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# 訓練データセットをロード
# root: データセットのダウンロード先と保存先のディレクトリ
# train: 訓練データセットを使用する場合はTrue
# download: データセットが存在しない場合にダウンロードする場合はTrue
# transform: 上記で定義した前処理を適用
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

# 訓練データのデータローダーを作成
# batch_size: バッチサイズ
# shuffle: 訓練データをシャッフルする場合はTrue
# num_workers: データローディングのためのサブプロセスの数
train_loader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True, num_workers=2)

# 検証データセットをロード（設定は訓練データと同様、ただしtrain=False）
valset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

# 検証データのデータローダーを作成（設定は訓練データと同様、ただしshuffle=False）
val_loader = torch.utils.data.DataLoader(valset, batch_size=32,
                                         shuffle=False, num_workers=2)

# テストデータセットをロード（設定は検証データと同様）
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

# テストデータのデータローダーを作成（設定は検証データと同様）
testloader = torch.utils.data.DataLoader(valset, batch_size=32,
                                         shuffle=False, num_workers=2)

# CIFAR-10のクラス名を定義
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# データローダーを辞書オブジェクトにまとめる（訓練と検証のデータローダー）
dataloaders_dict = {"train": train_loader, "val": val_loader}

In [ ]:
# 画像を表示する関数
def imshow(img):
    img = img / 2 + 0.5     # 正規化を解除
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# トレーニング画像をランダムに取得する
dataiter = iter(train_loader)
images, labels = next(dataiter)

# 画像表示
imshow(torchvision.utils.make_grid(images))
# ラベル表示
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

# ファインチューニング

In [ ]:
# 転移学習
net = torchvision.models.vgg16(pretrained=True)

In [ ]:
# 転移学習
# net = torchvision.models.wide_resnet50_2(pretrained=True)

In [ ]:
print(net)

In [ ]:
# netのclassifierの7番目の層（インデックスは6）の入力特徴量の数を取得
num_features = net.classifier[6].in_features
print(num_features)  # 4096 は、この層の入力特徴量の数を示しています。

# 最終層を新しい線形層に書き換えます。
# in_featuresは元の層の入力特徴量の数を保持し、out_featuresは新しい出力特徴量の数を設定します（ここでは10）。
net.classifier[6] = nn.Linear(in_features=num_features, out_features=10)
print(net)    # netの構造を出力し、最終層が正しく書き換えられたことを確認します。

In [ ]:
# 最適化手法を設定
# ファインチューニングで学習させるパラメータを、変数params_to_updateの1～3に格納する

# 各層のパラメータを格納するためのリストを初期化
params_to_update_1 = []
params_to_update_2 = []
params_to_update_3 = []

# 学習させる層のパラメータ名を指定
# update_param_names_1は、features層のパラメータを対象としています
# update_param_names_2とupdate_param_names_3は、classifier層の特定のパラメータを対象としています
update_param_names_1 = ["features"]
update_param_names_2 = ["classifier.0.weight",
                        "classifier.0.bias", "classifier.3.weight", "classifier.3.bias"]
update_param_names_3 = ["classifier.6.weight", "classifier.6.bias"]

# ネットワークの全てのパラメータをループし、指定された名前に基づいて各リストに格納する
for name, param in net.named_parameters():
    if update_param_names_1[0] in name:
        param.requires_grad = True          # 勾配計算を有効にし、この層を学習可能にします
        params_to_update_1.append(param)    # params_to_update_1リストにパラメータを追加します
        print("params_to_update_1に格納：", name)

    elif name in update_param_names_2:
        param.requires_grad = True          # 勾配計算を有効にし、この層を学習可能にします
        params_to_update_2.append(param)    # params_to_update_2リストにパラメータを追加します
        print("params_to_update_2に格納：", name)

    elif name in update_param_names_3:
        param.requires_grad = True          # 勾配計算を有効にし、この層を学習可能にします
        params_to_update_3.append(param)    # params_to_update_3リストにパラメータを追加します
        print("params_to_update_3に格納：", name)

    else:
        param.requires_grad = False         # 他の層では勾配計算を無効にし、学習を行いません
        print("勾配計算なし。学習しない：", name)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
# 最適化手法の設定
# SGD（確率的勾配降下法）を使用し、各パラメータのグループに対して異なる学習率を設定します。
# これにより、ネットワークの異なる部分を異なる速度で学習することができます。
optimizer = optim.SGD([
    {'params': params_to_update_1, 'lr': 1e-4},   # params_to_update_1のパラメータは学習率1e-4で学習されます
    {'params': params_to_update_2, 'lr': 5e-4},   # params_to_update_2のパラメータは学習率5e-4で学習されます
    {'params': params_to_update_3, 'lr': 1e-3}    # params_to_update_3のパラメータは学習率1e-3で学習されます
], momentum=0.9)

In [ ]:
def train_model(net, dataloaders_dict, criterion, optimizer, num_epochs):

    # 初期設定
    # GPUが使えるかを確認
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("使用デバイス：", device)

    # ネットワークをGPUへ
    net.to(device)

    # ネットワークがある程度固定であれば、高速化させる
    torch.backends.cudnn.benchmark = True

    # epochのループ
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch+1, num_epochs))
        print('-------------')

        # epochごとの訓練と検証のループ
        for phase in ['train', 'val']:
            if phase == 'train':
                net.train()  # モデルを訓練モードに
            else:
                net.eval()   # モデルを検証モードに

            epoch_loss = 0.0  # epochの損失和
            epoch_corrects = 0  # epochの正解数

            # 未学習時の検証性能を確かめるため、epoch=0の訓練は省略
            if (epoch == 0) and (phase == 'train'):
                continue

            # データローダーからミニバッチを取り出すループ
            for inputs, labels in tqdm(dataloaders_dict[phase]):

                # GPUが使えるならGPUにデータを送る
                inputs = inputs.to(device)
                labels = labels.to(device)

                # optimizerを初期化
                optimizer.zero_grad()

                # 順伝搬（forward）計算
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)  # 損失を計算
                    _, preds = torch.max(outputs, 1)  # ラベルを予測

                    # 訓練時はバックプロパゲーション
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                    # 結果の計算
                    epoch_loss += loss.item() * inputs.size(0)  # lossの合計を更新
                    # 正解数の合計を更新
                    epoch_corrects += torch.sum(preds == labels.data)

            # epochごとのlossと正解率を表示
            epoch_loss = epoch_loss / len(dataloaders_dict[phase].dataset)
            epoch_acc = epoch_corrects.double(
            ) / len(dataloaders_dict[phase].dataset)

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

In [ ]:
# 学習・検証を実行する
num_epochs=5
train_model(net, dataloaders_dict, criterion, optimizer, num_epochs=num_epochs)

## 🔧 実践問題1：ベースモデルを変更してファインチューニングする

ここまではVGG16をベースモデルとして使用しました。
PyTorchの `torchvision.models` には多数の学習済みモデルが用意されており、ベースモデルの選択が精度と計算コストに大きく影響します。

---

**問題：** 以下のコードの `______` 部分を埋めて、別のモデルでファインチューニングを行ってください。

ただし、モデルによって最終層の **属性名** と **入力次元数** が異なるため、自分で調べて正しく差し替える必要があります。
`print(net)` でモデル構造を確認し、最終全結合層がどの属性にあるか・入力次元がいくつかを把握してから穴埋めに取り組んでください。

| モデル | 最終層の属性名 | 入力次元 |
|:---|:---|:---|
| VGG16 | `classifier[6]` | ? |
| ResNet18 | ? | ? |
| ResNet50 | ? | ? |

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
まず <code>net = torchvision.models.????(pretrained=True)</code> でモデルを読み込み、<code>print(net)</code> で構造を確認しましょう。最終層の <code>in_features</code> を確認してから差し替えます。
</blockquote>

</details>

<br/>

In [ ]:
# 別のモデルでファインチューニングに挑戦する

# TODO: 使用するモデルを選んで読み込んでください
net_new = torchvision.models.______(pretrained=True)

# まずモデル構造を確認する
print(net_new)


In [ ]:
# TODO: 最終全結合層をCIFAR-10用（10クラス）に差し替えてください
net_new.______ = nn.Linear(in_features=______, out_features=10)

<details><summary>解答例</summary>

```python
# ResNet18の場合
net_new = torchvision.models.resnet18(pretrained=True)
net_new.fc = nn.Linear(in_features=512, out_features=10)

# ResNet50の場合
net_new = torchvision.models.resnet50(pretrained=True)
net_new.fc = nn.Linear(in_features=2048, out_features=10)
```

VGG系は `classifier` というSequentialの中に全結合層があり、ResNet系は `fc` という単一の属性に最終層があります。
また、入力次元もモデルにより異なります（VGG16: 4096、ResNet18: 512、ResNet50: 2048）。
モデルを切り替える際は必ず `print(net)` で構造を確認し、正しい属性名と次元数を把握する習慣をつけましょう。
</details>

In [ ]:
# 保存するファイルのパスを指定します。
PATH = './cifar_net.pth'

# torch.saveを使用してネットワークの状態（パラメータ）をファイルに保存します。
# net.state_dict()は、ネットワークのパラメータを含む辞書を返します。
torch.save(net.state_dict(), PATH)

In [ ]:
# iter関数を使用して、testloaderからイテレータを作成します。
dataiter = iter(testloader)

# nextメソッドを使用して、次のデータバッチを取得します。
# imagesはバッチ内の画像を、labelsは対応するラベルを格納します。
images, labels = next(dataiter)

# 画像の結果を表示する
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))

In [ ]:
# 保存されたモデルのパラメータをロードします。
# torch.loadは、指定されたパス（PATH）からPyTorchのモデルパラメータをロードします。
# load_state_dictは、ロードされたパラメータを現在のネットワーク（net）に適用します。
net.load_state_dict(torch.load(PATH))

In [ ]:
# torch.deviceを使用して、利用可能な場合はCUDAデバイスを、利用できない場合はCPUを選択します。
# torch.cuda.is_available()は、CUDAが利用可能かどうかをチェックします。
# "cuda:0"は、最初のCUDAデバイスを指定します。CUDAデバイスが利用できない場合、"cpu"を使用します。
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# データとラベルを選択されたデバイス（CUDAまたはCPU）に移動
images = images.to(device)
labels = labels.to(device)

In [ ]:
# ネットワークを通じて画像バッチを伝播させ、出力を取得します。
# 出力は、各画像に対するクラススコアを含むテンソルです。
outputs = net(images)

In [ ]:
# torch.max関数を使用して、各画像に対する最大のクラススコアを持つインデックスを取得します。
# このインデックスは、予測されたクラスのラベルに対応します。
_, predicted = torch.max(outputs, 1)

# 予測されたクラスのラベルを表示する。
# ここでは、バッチの最初の4つの画像の予測のみを表示しています。
print('Predicted: ', ' '.join('%5s' % classes[predicted[j]]
                              for j in range(4)))

In [ ]:
# 正解のカウントを初期化
correct = 0
# 合計のカウントを初期化
total = 0

# torch.no_grad()は、このコンテキスト内で勾配の計算を無効にし、
# メモリ使用量を削減し、速度を向上させるために使用されます（評価モード）
with torch.no_grad():
    # testloaderからデータバッチを取得し、それぞれをループします。
    for data in testloader:
        # データとラベルを取得
        images, labels = data

        # データとラベルを選択されたデバイス（CUDAまたはCPU）に移動
        images = images.to(device)
        labels = labels.to(device)

        # ネットワークを通じてイメージを渡し、出力を得る
        outputs = net(images)

        # 出力から、最大スコアを持つクラスを取得（予測）
        _, predicted = torch.max(outputs.data, 1)

        # totalにバッチのサイズを追加（評価するアイテムの合計数を更新）
        total += labels.size(0)

        # 正解の予測の数をカウントし、correctを更新
        correct += (predicted == labels).sum().item()

# ネットワークの正確さを計算し、テストイメージの10000枚に対する正確さを表示
print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

In [ ]:
# 各クラスでの正解数と合計数をトラックするためのリストを初期化します。
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))

# 勾配の計算を無効にすることで、評価モードに入ります（メモリを節約し、速度を向上させます）。
with torch.no_grad():
    # testloaderからデータバッチを取得し、それぞれをループします。
    for data in testloader:
        # データとラベルを取得
        images, labels = data

        # データとラベルを選択されたデバイス（CUDAまたはCPU）に移動します。
        images = images.to(device)
        labels = labels.to(device)

        # ネットワークを通じてイメージを渡し、出力を得ます。
        outputs = net(images)

        # 出力から最大スコアを持つクラスを取得します（予測）。
        _, predicted = torch.max(outputs, 1)

        # 正解の予測を見つける
        c = (predicted == labels).squeeze()

        # バッチ内の各サンプルに対してループし、クラスごとの正解数と合計数を更新します。
        # ここでは、バッチの最初の4つのサンプルのみを考慮しています。
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()   # 正しい予測を更新
            class_total[label] += 1               # クラスごとの合計を更新

# 各クラスの正確さを計算し、表示します。
for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        classes[i], 100 * class_correct[i] / class_total[i]))

## 🔧 実践問題2：Data Augmentationを追加して精度向上を狙う

ここまでの前処理は `ToTensor` と `Normalize` のみでした。
ファインチューニングにおいても、Data Augmentation（データ拡張）を加えることで汎化性能を向上させることができます。

---

**問題：** 以下のコードの `______` を埋めて、訓練データに対してランダムな水平反転と、ランダムに切り出してからリサイズする処理を追加してください。`transforms` モジュールの中から適切なクラスを選んで使う必要があります。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
水平反転には <code>transforms.Random______Flip()</code>、ランダム切り出し+リサイズには <code>transforms.Random______Crop()</code> が使えます。引数には出力サイズを指定します。CIFAR-10の画像サイズは32×32です。
</blockquote>

</details>

<br/>


In [ ]:
# Data Augmentationを追加した前処理を定義する

# TODO: ______を埋めて、ランダム水平反転とランダムクロップを追加してください
transform_train = transforms.Compose([
    transforms.______,              # ランダムな水平反転
    transforms.______,              # ランダムクロップ（パディング4px付き、出力32x32）
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Augmentation付きで訓練データを再読み込み
trainset_aug = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
train_loader_aug = torch.utils.data.DataLoader(
    trainset_aug, batch_size=64, shuffle=True, num_workers=2)

print(f'Augmentation付き訓練データ: {len(trainset_aug)}枚')
print(f'適用される変換: {transform_train}')

<details><summary>解答例</summary>

```python
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
```

`RandomHorizontalFlip()` はデフォルトで50%の確率で左右反転します。
`RandomCrop(32, padding=4)` は画像の周囲に4ピクセルのパディングを加えてから32×32をランダムに切り出すことで、位置のばらつきを生み出します。
これらを加えることでモデルが位置や向きの変動に頑健になり、特にデータが少ない場合に汎化性能の向上が期待できます。
なお、テストデータにはAugmentationを適用しないのが一般的です。
</details>

# 📝 確認演習問題
以下の各問題について、（あ）（い）に当てはまる語句の正しい組み合わせを1〜4から選んでください。

---

## 問題1：転移学習の考え方

転移学習とは、ある大規模データセットで学習済みのモデルの **（あ）** を別のタスクに流用する手法である。
特に、学習済みの重みを初期値として新しいタスクに合わせて再学習することを **（い）** と呼ぶ。

```
1. （あ）重み（パラメータ）　（い）ファインチューニング
2. （あ）損失関数　　　　　　（い）ファインチューニング
3. （あ）重み（パラメータ）　（い）蒸留
4. （あ）損失関数　　　　　　（い）蒸留
```

<details><summary>正解</summary>

**1.（あ）重み（パラメータ）（い）ファインチューニング**

転移学習では事前学習で獲得した重み（特徴表現）を活用し、ファインチューニングではそれを初期値として新タスク用にパラメータを微調整します。蒸留は大きなモデルの知識を小さなモデルに転写する別の手法です。
</details>

---

## 問題2：VGG16の最終層の差し替え

VGG16をCIFAR-10（10クラス）の分類に使う場合、元の最終全結合層は **（あ）** クラス分類用のため、
出力ユニット数を **（い）** に変更した新しい `nn.Linear` 層に差し替える必要がある。

```
1. （あ）100　　（い）10
2. （あ）1000　 （い）10
3. （あ）1000　 （い）100
4. （あ）10000　（い）10
```

<details><summary>正解</summary>

**2.（あ）1000（い）10**

VGG16はImageNet（1000クラス）で事前学習されているため、最終層の出力は1000です。CIFAR-10に適用するには出力を10に変更します。入力側の次元（4096）はそのまま維持します。
</details>

---

## 問題3：ファインチューニングの学習率戦略

ファインチューニングでは、事前学習済みの浅い層は汎用的な特徴を既に獲得しているため学習率を **（あ）** 設定し、
新しく追加した層はランダム初期化の状態から学習するため学習率を **（い）** 設定するのが一般的である。

```
1. （あ）高く　（い）低く
2. （あ）低く　（い）低く
3. （あ）低く　（い）高く
4. （あ）高く　（い）高く
```

<details><summary>正解</summary>

**3.（あ）低く（い）高く**

事前学習済みの層は既に有用な特徴を捉えているため、大きく変化させないよう小さい学習率にします。一方、新規追加層はゼロから学習する必要があるため、大きい学習率を設定して効率的に収束させます。
</details>

---

## 問題4：モデルの保存と復元

PyTorchでモデルを保存する際、推奨される方法はモデルオブジェクト全体ではなく **（あ）** を保存することである。
復元時には、まず同じアーキテクチャのモデルを定義してから **（い）** で読み込む。

```
1. （あ）state_dict()　　（い）load_state_dict()
2. （あ）parameters()　　（い）load_parameters()
3. （あ）state_dict()　　（い）import_weights()
4. （あ）optimizer　　　 （い）load_state_dict()
```

<details><summary>正解</summary>

**1.（あ）state_dict()（い）load_state_dict()**

`state_dict()` はモデルの全パラメータを辞書形式で返します。モデル構造は保存されないため、復元時は同じネットワーク定義を用意し `load_state_dict()` でパラメータだけを読み込みます。
</details>

---

## 問題5：推論時の勾配無効化

学習済みモデルでテストデータを推論する際、`with torch.no_grad():` を使うことで **（あ）** の計算が無効化される。
これにより **（い）** の使用量が削減され、推論速度も向上する。

```
1. （あ）勾配　（い）メモリ
2. （あ）損失　（い）メモリ
3. （あ）勾配　（い）ディスク
4. （あ）損失　（い）ディスク
```

<details><summary>正解</summary>

**1.（あ）勾配（い）メモリ**

`torch.no_grad()` は自動微分の計算グラフ構築を停止します。逆伝播用の中間値を保持しなくなるためメモリ消費が減り、計算自体も高速になります。
</details>